# CALIDAD DE DATOS PYSPARK — PROYECTO FÓRMULA 100


Este notebook replica la estructura del ejercicio original de calidad de datos, pero utiliza el archivo `dataset_maestro_f1.csv` del proyecto integrador sobre eficiencia de pit stops en Fórmula 1 (2011–2024).

## PARTE 1: PERFILADO — ENTENDER LOS PROBLEMAS

El objetivo es examinar la estructura y la calidad del dataset sin modificar todavía los datos. Se revisan las dimensiones, el esquema, los encabezados, los valores nulos y la distribución de las columnas categóricas.

### 1.1 Inicializar Spark y cargar el dataset en un DataFrame

Se utiliza el dataset maestro porque integra, para cada escudería y carrera, las variables operativas de pit stops, los puntos obtenidos, la posición en el campeonato y la variable objetivo `target_top5`.

In [1]:
# Importar las herramientas necesarias de PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Importar librerías auxiliares de Python
import time

# Guardar el momento de inicio para medir el tiempo de lectura
inicio = time.time()

# Crear o recuperar una sesión de Spark
spark = (
    SparkSession.builder
    .appName("calidadDatosF1")
    .getOrCreate()
)

# Leer el archivo CSV; el dataset maestro utiliza coma como separador
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .csv('../data/dataset_maestro_f1.csv')
)

# Contar la cantidad de registros
total_registros = df.count()

# Mostrar el tiempo utilizado para cargar el archivo
print(f"CSV leído en {time.time() - inicio:.1f} segundos")

# Mostrar las dimensiones del DataFrame
print(f"Filas sin corregir: {total_registros:,} | Columnas: {len(df.columns)}")

# Mostrar los tipos de datos inferidos por Spark
print("\nEsquema del DataFrame:")
df.printSchema()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 16:52:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


CSV leído en 4.8 segundos
Filas sin corregir: 2,923 | Columnas: 10

Esquema del DataFrame:
root
 |-- year: integer (nullable = true)
 |-- race_name: string (nullable = true)
 |-- constructor_name: string (nullable = true)
 |-- total_stops: integer (nullable = true)
 |-- mean_pit_stop: double (nullable = true)
 |-- delta_pit_stop: double (nullable = true)
 |-- race_points: double (nullable = true)
 |-- season_points: double (nullable = true)
 |-- championship_standing: double (nullable = true)
 |-- target_top5: integer (nullable = true)



### Descripción del Esquema Inferido
Al observar el esquema generado por `printSchema()`, podemos confirmar que Spark asignó correctamente los tipos de datos:
1. **`total_stops` (integer):** Se infirió como número entero, lo cual es correcto ya que representa conteos discretos de paradas en boxes durante la carrera.
2. **`mean_pit_stop` (double):** Se infirió como decimal (punto flotante), lo cual es el formato adecuado para medir el tiempo promedio de las paradas en segundos con alta precisión.
3. **`constructor_name` (string):** Spark reconoció correctamente esta columna como cadena de texto, ya que alberga los nombres de las escuderías.

### 1.2 Normalizar encabezados

Los espacios, puntos, tildes y caracteres especiales pueden generar problemas al escribir consultas en PySpark. Por ese motivo se transforman los nombres a minúsculas y formato `snake_case`.

In [2]:
print('Encabezado sin normalizar:')
print(df.columns)

nuevos_encabezados = [col.replace('.', '').replace(' ', '_') for col in df.columns]
print(f'\n Lista con nuevos encabezados: {nuevos_encabezados}')

# Asignar los nuevos encabezados al DataFrame
df= df.toDF(*nuevos_encabezados)
print('\n Nuevos encabezados normalizados agregados al DataFrame:')
print(df.columns)


Encabezado sin normalizar:
['year', 'race_name', 'constructor_name', 'total_stops', 'mean_pit_stop', 'delta_pit_stop', 'race_points', 'season_points', 'championship_standing', 'target_top5']

 Lista con nuevos encabezados: ['year', 'race_name', 'constructor_name', 'total_stops', 'mean_pit_stop', 'delta_pit_stop', 'race_points', 'season_points', 'championship_standing', 'target_top5']

 Nuevos encabezados normalizados agregados al DataFrame:
['year', 'race_name', 'constructor_name', 'total_stops', 'mean_pit_stop', 'delta_pit_stop', 'race_points', 'season_points', 'championship_standing', 'target_top5']


### 1.3 Perfilado de nulos

Los valores ausentes más comunes son:

- **Tipo 1:** valores `None` o `NULL`, detectados mediante `isNull()`.
- **Tipo 2:** cadenas vacías, detectadas comparando el contenido con `""`.

El perfilado permite identificar en qué columnas existen problemas y qué porcentaje del dataset representan.

In [3]:
print("=== PERFILADO DE NULOS ===")
print(f'{"Columna":<28} {"Nulos reales":>14} {"% nulos":>10} {"Strings vacíos":>18} {"% vacíos":>10}')
print("-" * 86)

# Crear un diccionario con el tipo de dato de cada columna
tipos_columnas = dict(df.dtypes)
for columna in df.columns:
    # Contar valores NULL
    nulos_reales = df.filter(F.col(columna).isNull()).count()

    # Inicializar el conteo de cadenas vacías
    strings_vacios = 0

    # Las cadenas vacías solo pueden buscarse en columnas de texto
    if tipos_columnas[columna] == "string":
        strings_vacios = df.filter(F.trim(F.col(columna)) == "").count()

    # Calcular porcentajes con relación al total de registros
    porcentaje_nulos = (nulos_reales / total_registros) * 100
    porcentaje_vacios = (strings_vacios / total_registros) * 100
    print(
        f"{columna:<28} "
        f"{nulos_reales:>14,} "
        f"{porcentaje_nulos:>9.2f}% "
        f"{strings_vacios:>18,} "
        f"{porcentaje_vacios:>9.2f}%"
    )


=== PERFILADO DE NULOS ===
Columna                        Nulos reales    % nulos     Strings vacíos   % vacíos
--------------------------------------------------------------------------------------
year                                      0      0.00%                  0      0.00%
race_name                                 0      0.00%                  0      0.00%
constructor_name                          0      0.00%                  0      0.00%
total_stops                               0      0.00%                  0      0.00%
mean_pit_stop                             0      0.00%                  0      0.00%
delta_pit_stop                            0      0.00%                  0      0.00%
race_points                               0      0.00%                  0      0.00%
season_points                             1      0.03%                  0      0.00%
championship_standing                     1      0.03%                  0      0.00%
target_top5                         

### Justificación de Limpieza e Interpretación de Nulos
Durante el perfilado de calidad, se identificaron nulos reales en las columnas `season_points` y `championship_standing` (un 0.03% del total). Se determinó que corresponden a **registros corruptos** y no a nulos estructurales, ya que no es lógicamente posible que una escudería participe en una carrera sin registrar una posición o puntos acumulados en el campeonato (como se evidenció con el registro aislado de Sauber en Australia 2011). 

Por lo tanto, se justifica su eliminación total del dataset mediante filtros estrictos para evitar sesgos numéricos en el análisis predictivo. Por el resto de columnas se pudo evidenciar la aucencia de nulos por lo que no sera necesario aplicar más cambios en este aspecto



### 1.4 Perfilado de columnas categóricas

Las columnas categóricas representan valores pertenecientes a un dominio determinado. En este proyecto se revisan:

- `race_name`: nombre del Gran Premio.
- `constructor_name`: nombre de la escudería.
- `target_top5`: clase binaria, donde 1 significa Top 5 y 0 significa fuera del Top 5.

El conteo permite detectar categorías inesperadas, errores de escritura y desbalance entre clases.

In [4]:
print("=== VALORES ÚNICOS PARA COLUMNAS CATEGÓRICAS ===")

# Seleccionar las variables categóricas relevantes del proyecto
columnas_categoricas = ["race_name", "constructor_name", "target_top5"]

# Agrupar y contar la frecuencia de cada categoría
for columna in columnas_categoricas:
    print(f"\n-- Columna: {columna} --")
    (df.groupBy(columna).count().orderBy("count", ascending= False).show(5, truncate=False))


=== VALORES ÚNICOS PARA COLUMNAS CATEGÓRICAS ===

-- Columna: race_name --
+--------------------+-----+
|race_name           |count|
+--------------------+-----+
|Spanish Grand Prix  |145  |
|Abu Dhabi Grand Prix|145  |
|Hungarian Grand Prix|144  |
|Italian Grand Prix  |143  |
|British Grand Prix  |142  |
+--------------------+-----+
only showing top 5 rows


-- Columna: constructor_name --
+----------------+-----+
|constructor_name|count|
+----------------+-----+
|Williams        |283  |
|Mercedes        |282  |
|Red Bull        |282  |
|McLaren         |281  |
|Ferrari         |279  |
+----------------+-----+
only showing top 5 rows


-- Columna: target_top5 --
+-----------+-----+
|target_top5|count|
+-----------+-----+
|0          |1517 |
|1          |1406 |
+-----------+-----+



In [5]:
print("=== PERFILADO CATEGÓRICO SECUENCIAL ===")

# Aplicando la secuencia: Filtro -> Agrupo -> Cuento -> Ordeno -> Muestro
# Analizamos cuáles fueron las escuderías con más participaciones desde el año 2020
(
    df.filter(F.col("year") >= 2020)                   # 1. FILTRO
    .groupBy("constructor_name")                       # 2. AGRUPO
    .count()                                           # 3. CUENTO
    .orderBy("count", ascending=False)                 # 4. ORDENO
    .show(5, truncate=False)                           # 5. MUESTRO
)

=== PERFILADO CATEGÓRICO SECUENCIAL ===
+----------------+-----+
|constructor_name|count|
+----------------+-----+
|Williams        |105  |
|Mercedes        |105  |
|Red Bull        |105  |
|McLaren         |103  |
|Ferrari         |103  |
+----------------+-----+
only showing top 5 rows



# PARTE 2: LIMPIEZA Y TRANSFORMACIÓN — CORREGIR PROBLEMAS

Después del perfilado se aplican reglas de calidad sobre las variables indispensables para el análisis. El objetivo es conservar únicamente observaciones completas y coherentes.

### 2.1 Eliminar registros corruptos

Se eliminan registros que no puedan utilizarse correctamente en los análisis del proyecto:

- `year`, `race_name` y `constructor_name` no pueden ser nulos.
- `race_name` y `constructor_name` no pueden ser cadenas vacías.
- `season_points` y `championship_standing` deben tener un valor.
- `target_top5` solo puede tomar los valores 0 o 1.

En el dataset original se detectó un registro de Sauber en el Gran Premio de Australia de 2011 sin puntos acumulados ni posición en el campeonato.

In [6]:
# Construir una condición que representa un registro válido
condicion_valida = (
    F.col("year").isNotNull()
    & F.col("race_name").isNotNull()
    & (F.trim(F.col("race_name")) != "")
    & F.col("constructor_name").isNotNull()
    & (F.trim(F.col("constructor_name")) != "")
    & F.col("season_points").isNotNull()
    & F.col("championship_standing").isNotNull()
    & F.col("target_top5").isin(0, 1)
)

# Conservar únicamente los registros que cumplen las reglas de calidad
df_limpio = df.filter(condicion_valida)

# Identificar los registros eliminados para mantener trazabilidad
df_corruptos = df.filter(~condicion_valida)

# Contar resultados antes y después de la limpieza
registros_limpios = df_limpio.count()
registros_eliminados = total_registros - registros_limpios

print(f"Registros originales con errores: {total_registros:,}")
print(f"Registros eliminados: {registros_eliminados:,}")
print(f"Registros limpios: {registros_limpios:,}")

# Mostrar los registros rechazados
print("\nRegistros eliminados por incumplir las reglas de calidad:")
df_corruptos.show(truncate=False)


Registros originales con errores: 2,923
Registros eliminados: 1
Registros limpios: 2,922

Registros eliminados por incumplir las reglas de calidad:
+----+---------------------+----------------+-----------+-------------+--------------+-----------+-------------+---------------------+-----------+
|year|race_name            |constructor_name|total_stops|mean_pit_stop|delta_pit_stop|race_points|season_points|championship_standing|target_top5|
+----+---------------------+----------------+-----------+-------------+--------------+-----------+-------------+---------------------+-----------+
|2011|Australian Grand Prix|Sauber          |3          |23.898       |-0.17         |0.0        |NULL         |NULL                 |0          |
+----+---------------------+----------------+-----------+-------------+--------------+-----------+-------------+---------------------+-----------+



### Transformación de Mayúsculas y cateogorización


In [7]:
print("=== TRANSFORMACIONES: MAYÚSCULAS Y CATEGORIZACIÓN ===")

# Transformación 1: Normalizar la columna 'race_name' a mayúsculas
df_transformado = df_limpio.withColumn(
    "race_name_upper", 
    F.upper(F.col("race_name"))
)

# Transformación 2: Crear columna clasificatoria de eficiencia del pit stop con 3 condiciones
# Clasificamos el tiempo medio de pit stop ('mean_pit_stop') en categorías operativas de la F1
df_transformado = df_transformado.withColumn(
    "pit_stop_efficiency",
    F.when(F.col("mean_pit_stop") < 23.5, "Rápido")
     .when((F.col("mean_pit_stop") >= 23.5) & (F.col("mean_pit_stop") <= 25.0), "Promedio")
     .otherwise("Lento")
)

# Mostrar una muestra de los resultados de ambas transformaciones
df_transformado.select(
    "race_name", 
    "race_name_upper", 
    "mean_pit_stop", 
    "pit_stop_efficiency"
).show(10, truncate=False)

=== TRANSFORMACIONES: MAYÚSCULAS Y CATEGORIZACIÓN ===
+---------------------+---------------------+-------------+-------------------+
|race_name            |race_name_upper      |mean_pit_stop|pit_stop_efficiency|
+---------------------+---------------------+-------------+-------------------+
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|23.763       |Promedio           |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|21.855       |Rápido             |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|24.1         |Promedio           |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|24.578       |Promedio           |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|25.23        |Lento              |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|24.761       |Promedio           |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|24.871       |Promedio           |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|25.166       |Lento              |
|Australian Grand Prix|AUSTRALIAN GRAND PRIX|22.281       |Rápido 